# Stage 36: exact-parity parallel Stage 18 benchmark

Stage 18の予測内容は変更せず、well単位の直列1 workerと並列4 workersを比較します。CSV bytesが完全一致し、hidden 200 wellsの保守的推定が10分以内のときだけv4 packageをKaggleへ持ち込みます。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess
REPOSITORY_URL = 'https://github.com/Okada-N13/rogii.git'
repo_dir = Path('/content/ROGII')
drive_root = Path('/content/drive/MyDrive/kaggle/rogii')
artifact_dir = drive_root / 'artifacts'
data_dir = drive_root / 'data'
if not (repo_dir / '.git').is_dir():
    subprocess.run(['git', 'clone', REPOSITORY_URL, str(repo_dir)], check=True)
else:
    subprocess.run(['git', '-C', str(repo_dir), 'pull', '--ff-only', 'origin', 'main'], check=True)
if shutil.which('uv') is None:
    subprocess.run(['bash', '-lc', 'curl -LsSf https://astral.sh/uv/install.sh | sh'], check=True)
os.environ['PATH'] = '/root/.local/bin:' + os.environ['PATH']
subprocess.run(['uv', 'sync', '--frozen'], cwd=repo_dir, check=True)
artifact_dir.mkdir(parents=True, exist_ok=True)
def run_checked(command):
    result = subprocess.run(command, cwd=repo_dir, text=True, capture_output=True)
    if result.stdout: print(result.stdout, flush=True)
    if result.returncode:
        print(result.stderr, flush=True)
        raise RuntimeError(f'command failed: {command}')
    return result
subprocess.run(['git', '-C', str(repo_dir), 'rev-parse', '--short', 'HEAD'], check=True)

## v4 package構築

Stage 18Dの学習結果と固定fold assignmentから、packed donor cacheと4-worker inferenceを含むpackageを構築します。GPUは不要です。

In [ ]:
stage16b_run = artifact_dir / 'stage16b_testlike_validation_full_v003'
stage18d_run = artifact_dir / 'stage18d_learned_donor_ranker_full_v001'
assert (stage16b_run / 'well_assignments.parquet').is_file(), stage16b_run
assert (stage18d_run / 'donor_training_rows.parquet').is_file(), stage18d_run
assert (data_dir / 'train').is_dir() and (data_dir / 'test').is_dir(), data_dir
PACKAGE_ID = 'stage36_parallel_retrieval_package_v001'
package_dir = artifact_dir / PACKAGE_ID
if not (package_dir / 'summary.json').is_file():
    if package_dir.exists(): shutil.rmtree(package_dir)
    run_checked(['uv','run','rogii-stage18-package','--stage16b-run',str(stage16b_run),'--stage18d-run',str(stage18d_run),'--data-dir',str(data_dir),'--artifact-dir',str(artifact_dir),'--run-id',PACKAGE_ID])
package_summary = json.loads((package_dir / 'summary.json').read_text())
assert package_summary['parallel_well_workers'] == 4, package_summary
package_summary

## 直列・並列完全一致benchmark

同じ3 public wellsへ直列と並列を適用します。推定時間は観測した並列時間を固定費として残し、残り197 wellsを直列の1-well平均時間で4分割する保守的な式です。

In [ ]:
RUN_ID = 'stage36_parallel_retrieval_benchmark_v001'
run_dir = artifact_dir / RUN_ID
if not (run_dir / 'summary.json').is_file():
    if run_dir.exists(): shutil.rmtree(run_dir)
    run_checked(['uv','run','rogii-stage36-retrieval-benchmark','--package-dir',str(package_dir),'--data-dir',str(data_dir),'--artifact-dir',str(artifact_dir),'--run-id',RUN_ID,'--workers','4','--hidden-wells','200','--runtime-gate-seconds','600'])
summary = json.loads((run_dir / 'summary.json').read_text())
summary

In [ ]:
keys = ['promoted_to_kaggle_package_v4','serial_elapsed_seconds','parallel_elapsed_seconds','observed_speedup','projected_hidden_elapsed_seconds','gates','next_step']
{key: summary[key] for key in keys}
print('Kaggle Dataset upload file:', package_summary['zip'])

最後の辞書を共有してください。`promoted_to_kaggle_package_v4=True`の場合だけzipをKaggle Datasetへ上書きアップロードし、更新済みnotebook 460を実行します。